This notebook is used to synthesize the trained models with HLS4MLs backend Vitis Unified. Some synthesis include bitfile-generation, and varies with strategy.

In [7]:
import os
import sys
import time
import numpy as np
import matplotlib.pyplot as plt
import keras
import tensorflow as tf
from sklearn.metrics import accuracy_score

# Load Vitis into path
os.environ['PATH'] = os.environ['XILINX_VITIS'] + '/bin:' + os.environ['PATH']

In [8]:
model_to_test = 'jettag-qkeras'

model_configs = [
    {
        "description": "qkeras acc=0.7298 Distributed Arithmetic",
        "model_revision": "qkeras",
        "keras_model_path": "jettag-qkeras/Qkeras_Pruned/model_Qkeras_Pruned_acc=0.7298.h5",
        "hls4ml_strategy": "DA",
        "hls4ml_generate_bitfile": True,
        "hls4ml_revision": "acc=0.7298_VU_DA_bitfile",
    },
    {
        "description": "qkeras acc=0.7298 latency",
        "model_revision": "qkeras",
        "keras_model_path": "jettag-qkeras/Qkeras_Pruned/model_Qkeras_Pruned_acc=0.7298.h5",
        "hls4ml_strategy": "latency",
        "hls4ml_generate_bitfile": True,
        "hls4ml_revision": "acc=0.7298_VU_latency_bitfile",
    },
]

In [9]:
# Load dataset which is preprocessed in another notebook
#X_train_val = np.load('Data/x_train_val.npy')
#X_test = np.load('Data/x_test.npy')
#y_train_val = np.load('Data/y_train_val.npy')
#y_test = np.load('Data/y_test.npy')
#classes = np.load('Data/classes.npy', allow_pickle=True)


In [10]:
import os

def prepare_directory(model_config):
    output_dir = os.path.join(
        os.path.dirname(os.path.abspath(model_config["keras_model_path"])),
        f"hls4ml_prj_{model_config['hls4ml_revision']}",
    )
    os.makedirs(output_dir, exist_ok=True)

    description = f"""
    Description of HLS4ML-project.

    {model_config['description']}

    - Bitfile: {model_config['hls4ml_generate_bitfile']}
    - Environment: devenv-vu-hgq+da (environment-HGQ+DA.yml)
    - Target Device: KV260 (xck26-sfvc784-2LV-c)
    - Dataset: HLS4ML LHC Jets
    - Vivado/Vitis: 2025.2
    - Model Architecture: {model_to_test}
    - Model Revision: {model_config['model_revision']}
    - HLS4ML Revision: {model_config['hls4ml_revision']}

    The model summary is in the parent-directory, `summary.txt`
    """
    with open(os.path.join(output_dir, "description.md"), "w", encoding="utf-8") as f:
        f.write(description)

    return output_dir

In [11]:
from qkeras.utils import load_qmodel

import hls4ml

def compile_model(keras_model_path, output_dir, hls4ml_strategy):
    # This replaces the standard keras load_model
    model = load_qmodel(keras_model_path)
    
    hls_config = hls4ml.utils.config_from_keras_model(model, granularity='name')
    
    strategy = 'Distributed Arithmetic' if hls4ml_strategy == 'DA' else hls4ml_strategy
    hls_config['Model']['Strategy'] = strategy # https://fastmachinelearning.org/hls4ml/api/configuration.html#top-level-configuration

    hls_model = hls4ml.converters.convert_from_keras_model( 
        model,    
        backend     =   'vitisunified',
        hls_config  =   hls_config,
        output_dir  =   output_dir, 
        board       =   'kv260',
        part        =   'xck26-sfvc784-2LV-c',
        clock_period=   '5',
        strategy    =   strategy
    )
    hls_model.compile()
    return hls_model

In [12]:
# Hotfix for crashing, see README
os.environ['LD_PRELOAD'] = '/lib/x86_64-linux-gnu/libudev.so.1'

process_model_durations = []

# Run through every model
for i, model_config in enumerate(model_configs):
    print(f"Processing model {i+1}/{len(model_configs)}: {model_config['description']}")
    print(f"%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%")
    start_time = time.time()
    output_dir = prepare_directory(model_config)
    
    hls_model = compile_model(
        model_config["keras_model_path"],
        output_dir,
        model_config["hls4ml_strategy"],
    )

    # Create complete bitfile (Vitis Unified-backend) or IP-block (Vitis-backend)
    hls_model.build(
        synth=True,
        bitfile=model_config["hls4ml_generate_bitfile"],
        csim=False # Simulation (CSIM and COSIM) needs input_data_tb and output_data_tb https://fastmachinelearning.org/hls4ml/autodoc/hls4ml.converters.html#hls4ml.converters.convert_from_keras_model
    )
    
    elapsed_time = time.time() - start_time
    process_model_durations.append({
        'model': model_config['description'],
        'time_seconds': elapsed_time,
        'time_minutes': elapsed_time / 60
    })
    print(f"\nModel {i+1} completed in {elapsed_time:.2f} seconds ({elapsed_time/60:.2f} minutes)")


Processing model 1/2: qkeras acc=0.7298 Distributed Arithmetic
%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%


/opt/miniconda3/envs/devenv-qkeras/lib/python3.10/site-packages/keras/src/initializers/initializers.py:120: UserWarning: The initializer LecunUniform is unseeded and being called multiple times, which will return identical values each time (even if the initializer is unseeded). Please update your code to provide a seed to the initializer, or avoid using the same initializer instance more than once.
  warnings.warn(
/opt/miniconda3/envs/devenv-qkeras/lib/python3.10/site-packages/keras/src/constraints.py:365: UserWarning: The `keras.constraints.serialize()` API should only be used for objects of type `keras.constraints.Constraint`. Found an instance of type <class 'qkeras.quantizers.quantized_bits'>, which may lead to improper serialization.
  warnings.warn(



****** v++ v2025.2 (64-bit)
  **** SW Build 6295257 on 2025-11-13-01:29:13
  **** Start of session at: Thu May  7 14:39:48 2026
    ** Copyright 1986-2022 Xilinx, Inc. All Rights Reserved.
    ** Copyright 2022-2025 Advanced Micro Devices, Inc. All Rights Reserved.

  **** HLS Build v2025.2 6295257
INFO: [HLS 200-2005] Using work_dir /work/development/jettag/jettag-qkeras/Qkeras_Pruned/hls4ml_prj_acc=0.7298_VU_DA_bitfile/vitis_workspace/myproject/vitis_unified_project 
INFO: [HLS 200-2176] Writing Vitis IDE component file /work/development/jettag/jettag-qkeras/Qkeras_Pruned/hls4ml_prj_acc=0.7298_VU_DA_bitfile/vitis_workspace/myproject/vitis_unified_project/vitis-comp.json
INFO: [HLS 200-10] Creating and opening component '/work/development/jettag/jettag-qkeras/Qkeras_Pruned/hls4ml_prj_acc=0.7298_VU_DA_bitfile/vitis_workspace/myproject/vitis_unified_project'.
INFO: [HLS 200-1505] Using default flow_target 'vivado'
Resolution: For help on HLS 200-1505 see docs.amd.com/access/sources/dit

In [18]:
import pandas as pd

timestamp = time.strftime("%Y%m%d_%H%M%S")
timing_df = pd.DataFrame(process_model_durations)
timing_df = timing_df[["model", "time_seconds", "time_minutes"]]
timing_df.to_csv(f"{model_to_test}/timing_summary_{timestamp}.csv", index=False)

display(timing_df)

total_time = timing_df["time_seconds"].sum()
average_time = timing_df["time_seconds"].mean()
print(f"\nTotal time: {total_time:.2f} seconds ({total_time/60:.2f} minutes)")
print(f"Average time per model: {average_time:.2f} seconds ({average_time/60:.2f} minutes)")

,model,time_seconds,time_minutes
0,qkeras acc=0.7298 Distributed Arithmetic,591.011794,9.850197
1,qkeras acc=0.7298 latency,556.897948,9.281632



Total time: 1147.91 seconds (19.13 minutes)
Average time per model: 573.95 seconds (9.57 minutes)


Copy bitfile and hwh from export-directory to directory for onboard-verification.
Make sure the driver and testdata is copied manually.

In [14]:
import glob
import shutil

target_dir = '../../onboard-verification/jettag/DUT'

for model_config in model_configs:
    export_dir = os.path.join(
        os.path.dirname(os.path.abspath(model_config["keras_model_path"])),
        f"hls4ml_prj_{model_config['hls4ml_revision']}",
        'export'
    )
    model_target_dir = os.path.join(
        target_dir,
        f"{model_config['model_revision']}_{model_config['hls4ml_revision']}"
    )
    os.makedirs(model_target_dir, exist_ok=True)

    for extension in ('bit', 'hwh'):
        matches = sorted(glob.glob(os.path.join(export_dir, f'*.{extension}')))
        if not matches:
            print(f"ERROR: No .{extension} file found in {export_dir}")
            continue
        shutil.copy2(matches[0], model_target_dir)
        print(f"Copied {matches[0]} -> {model_target_dir}")

Copied /work/development/jettag/jettag-qkeras/Qkeras_Pruned/hls4ml_prj_acc=0.7298_VU_DA_bitfile/export/system.bit -> ../../onboard-verification/jettag/DUT/qkeras_acc=0.7298_VU_DA_bitfile
Copied /work/development/jettag/jettag-qkeras/Qkeras_Pruned/hls4ml_prj_acc=0.7298_VU_DA_bitfile/export/system.hwh -> ../../onboard-verification/jettag/DUT/qkeras_acc=0.7298_VU_DA_bitfile
Copied /work/development/jettag/jettag-qkeras/Qkeras_Pruned/hls4ml_prj_acc=0.7298_VU_latency_bitfile/export/system.bit -> ../../onboard-verification/jettag/DUT/qkeras_acc=0.7298_VU_latency_bitfile
Copied /work/development/jettag/jettag-qkeras/Qkeras_Pruned/hls4ml_prj_acc=0.7298_VU_latency_bitfile/export/system.hwh -> ../../onboard-verification/jettag/DUT/qkeras_acc=0.7298_VU_latency_bitfile
